# HCMUTE Decision benchmark: tái tạo hình

Notebook này tái tạo tám hình từ số đo tổng hợp đã công bố: năm cấu hình, 158 mẫu giữ riêng, ba lượt/model. Không gọi model và không cần API key; có thể dùng runtime CPU. Muốn chạy lại suy luận trên log cần dataset riêng được cấp quyền như README.


In [ ]:
import subprocess
from pathlib import Path

SOURCE_REF = "decision-benchmark-20261106"
REPOSITORY = "https://github.com/AndyAnh174/dataops-control-plane.git"
REPO_DIR = Path("/content/dataops-decision-benchmark-20261106")
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", SOURCE_REF, REPOSITORY, str(REPO_DIR)],
        check=True,
    )
BENCH_ROOT = REPO_DIR / "benchmark" / "decision"
source_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
print("Source snapshot:", SOURCE_REF, source_commit)
print("Benchmark source:", BENCH_ROOT)

## Xuất hình và ZIP

Cài phụ thuộc vẽ vào thư mục riêng; xuất PNG 300 dpi, SVG, PDF từng hình và một PDF tám trang. Manifest ghi hash dữ liệu/script/file và các phiên bản thư viện.


In [ ]:
import sys

subprocess.run(
    [sys.executable, "-u", str(BENCH_ROOT / "render_colab.py")], cwd=BENCH_ROOT, check=True
)

In [ ]:
from IPython.display import Image, Markdown, display

for stem, title in [
    ("01_quality_heatmap", "Điểm tổng và heatmap F1"),
    ("02_latency_p50_p95", "Độ trễ p50/p95"),
    ("03_quality_latency", "Chất lượng và độ trễ"),
    ("04_probability_quality", "Brier và AUROC"),
    ("05_prompt_injection", "Probe chèn chỉ dẫn"),
    ("06_gpu_memory", "GPU peak quan sát"),
    ("07_benchmark_flow", "Luồng benchmark"),
    ("08_injection_design", "Thiết kế đối chứng"),
]:
    display(Markdown("## " + title))
    display(Image(filename=str(BENCH_ROOT / "figures" / (stem + ".png")), width=1100))

## Cách đọc kết quả

Chỉ số proxy là trung bình đều macro F1 prod/dev/CI, không phải accuracy hoặc Decision Index của bài tham khảo. Dev thiếu INFO. Prod/dev chỉ có một cụm nguồn trong test; ba lượt lặp không tăng số mẫu độc lập. Probe chỉ có sáu cặp/model. Chưa có nhãn incident/action được con người xác nhận, nên chưa kết luận ưu thế production hoặc khả năng chống tấn công tổng quát.


In [ ]:
from google.colab import files

archive = BENCH_ROOT / "HCMUTE-Decision-Charts-20261106.zip"
print("PDF 8 trang:", BENCH_ROOT / "figures" / "all_charts.pdf")
print("ZIP:", archive)
files.download(str(archive))